# 09. Inventory Optimization

## Objective

This notebook analyzes product inventory using transaction-level retail data.

The analysis identifies:
- Fast-moving products
- Slow-moving products
- High-value inventory using ABC Analysis

### Input

- online_retail_II_feature_engineered.csv

### Output

- inventory_analysis.csv

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [2]:
inventory_df = pd.read_csv(
    "../data/processed/online_retail_II_feature_engineered.csv"
)

print("Dataset loaded successfully!")

print("Shape:", inventory_df.shape)

display(inventory_df.head())

Dataset loaded successfully!
Shape: (779425, 20)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,MonthName,Quarter,Day,DayOfWeek,Hour,Revenue,TimeOfDay,Season,BasketSize,UniqueProducts
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,83.4,Morning,Winter,166,8
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,81.0,Morning,Winter,166,8
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,81.0,Morning,Winter,166,8
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,100.8,Morning,Winter,166,8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,30.0,Morning,Winter,166,8


## 3. Data Validation

In [3]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

inventory_df.info()

print("\nDataset Shape:", inventory_df.shape)

print("\nMissing Values:", inventory_df.isnull().sum().sum())

print("Duplicate Rows:", inventory_df.duplicated().sum())

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 779425 entries, 0 to 779424
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Invoice         779425 non-null  int64  
 1   StockCode       779425 non-null  str    
 2   Description     779425 non-null  str    
 3   Quantity        779425 non-null  int64  
 4   InvoiceDate     779425 non-null  str    
 5   Price           779425 non-null  float64
 6   Customer ID     779425 non-null  float64
 7   Country         779425 non-null  str    
 8   Year            779425 non-null  int64  
 9   Month           779425 non-null  int64  
 10  MonthName       779425 non-null  str    
 11  Quarter         779425 non-null  int64  
 12  Day             779425 non-null  int64  
 13  DayOfWeek       779425 non-null  str    
 14  Hour            779425 non-null  int64  
 15  Revenue         779425 non-null  float64
 16  TimeOfDay       779425 non-null  str    
 17  S

## 4. Product-Level Aggregation

In [4]:
inventory_analysis = (
    inventory_df.groupby(["StockCode", "Description"])
    .agg(
        TotalQuantity=("Quantity", "sum"),
        TotalRevenue=("Revenue", "sum"),
        TotalOrders=("Invoice", "nunique"),
        AveragePrice=("Price", "mean")
    )
    .reset_index()
)

print("Product-level aggregation completed successfully!")

print("Shape:", inventory_analysis.shape)

inventory_analysis.head()

Product-level aggregation completed successfully!
Shape: (5315, 6)


,StockCode,Description,TotalQuantity,TotalRevenue,TotalOrders,AveragePrice
0,10002,INFLATABLE POLITICAL GLOBE,8479,6638.27,297,0.840960
1,10080,GROOVY CACTUS INFLATABLE,303,124.61,26,0.509259
2,10109,BENDY COLOUR PENCILS,4,1.68,1,0.420000
3,10120,DOGGY RUBBER,648,136.08,62,0.210000
4,10123C,HEARTS WRAPPING TAPE,628,226.76,46,0.621739


## 5. ABC Inventory Analysis

In [5]:
# Sort products by revenue
inventory_analysis = inventory_analysis.sort_values(
    by="TotalRevenue",
    ascending=False
).reset_index(drop=True)

# Calculate cumulative revenue percentage
inventory_analysis["RevenuePercentage"] = (
    inventory_analysis["TotalRevenue"]
    / inventory_analysis["TotalRevenue"].sum()
) * 100

inventory_analysis["CumulativeRevenue"] = (
    inventory_analysis["RevenuePercentage"].cumsum()
)

print("ABC revenue calculations completed!")

inventory_analysis.head()

ABC revenue calculations completed!


,StockCode,Description,TotalQuantity,TotalRevenue,TotalOrders,AveragePrice,RevenuePercentage,CumulativeRevenue
0,22423,REGENCY CAKESTAND 3 TIER,24124,277656.25,3317,12.461649,1.598040,1.598040
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,91757,247048.01,4888,2.870666,1.421875,3.019915
2,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1,2.080000,0.969620,3.989535
3,M,Manual,9384,151777.67,620,214.785888,0.873550,4.863085
4,85099B,JUMBO BAG RED RETROSPOT,74224,134307.44,2612,1.973870,0.773001,5.636086


## 6. Inventory Classification

In [6]:
def classify_inventory(value):
    if value <= 80:
        return "A"
    elif value <= 95:
        return "B"
    else:
        return "C"

inventory_analysis["ABC_Class"] = (
    inventory_analysis["CumulativeRevenue"]
    .apply(classify_inventory)
)

print("ABC classification completed!")

print("\nABC Distribution:")
print(inventory_analysis["ABC_Class"].value_counts())

ABC classification completed!

ABC Distribution:
ABC_Class
C    2810
B    1367
A    1138
Name: count, dtype: int64


## 7. Product Movement Classification

In [7]:
# Classify products based on TotalQuantity
q75 = inventory_analysis["TotalQuantity"].quantile(0.75)
q25 = inventory_analysis["TotalQuantity"].quantile(0.25)

def movement_category(quantity):
    if quantity >= q75:
        return "Fast Moving"
    elif quantity >= q25:
        return "Medium Moving"
    else:
        return "Slow Moving"

inventory_analysis["MovementCategory"] = inventory_analysis["TotalQuantity"].apply(movement_category)

print("Movement categories created successfully!\n")

print(inventory_analysis["MovementCategory"].value_counts())

Movement categories created successfully!

MovementCategory
Medium Moving    2657
Fast Moving      1329
Slow Moving      1329
Name: count, dtype: int64


## 8. Save Inventory Analysis

In [8]:
inventory_analysis.to_csv(
    "../data/processed/inventory_analysis.csv",
    index=False
)

print("inventory_analysis.csv saved successfully!")

inventory_analysis.csv saved successfully!


## 9. Business Insights

### Key Findings

- Products were aggregated at the StockCode level.
- ABC Analysis identified high-value (A), medium-value (B), and low-value (C) inventory.
- Product movement was classified into Fast Moving, Medium Moving, and Slow Moving based on sales quantity.
- This analysis supports inventory planning, stock prioritization, and purchasing decisions.

## 10. Summary

This notebook successfully analyzed product inventory and generated inventory-level insights.

### Input
- online_retail_II_feature_engineered.csv

### Output
- inventory_analysis.csv

### Next Notebook
- 10_Dashboard_Dataset_Preparation.ipynb